In [1]:
import cwe_db

In [2]:

java_source = "C:\\Users\\Andrew\\OneDrive\\Documents\\Juliet Java 1.3\\src\\testcases"
c_source = "C:\\Users\\Andrew\\OneDrive\\Documents\\Juliet C_C++ 1.3\\testcases"
#cs_source = "C:\\Users\\Andrew\\OneDrive\\Documents\\Juliet C# 1.3\\src\\testcases\\* "
java_manifest = "manifests\\java_manifest.xml"
c_manifest = "manifests\\cmanifest.xml"

In [3]:
cwe_db.record("java_10+.db", java_manifest, java_source, min_lines=10)

In [4]:
cwe_db.record("c_10+.db", c_manifest, c_source, min_lines=10)

In [6]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from keras.utils import pad_sequences
import sentencepiece as spm
import sqlite3
from collections import Counter

def query_balanced_functions(db_path, min_lines=10, max_lines=50, max_per_class=None):
    """
    Query balanced vulnerable and non-vulnerable functions from the database.
    
    Args:
        db_path: Path to the SQLite database
        min_lines: Minimum function length
        max_lines: Maximum function length
        max_per_class: Maximum number of functions per class (None for no limit)
    
    Returns:
        List of tuples (code, vulnerability_label)
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Get vulnerable functions
    vuln_query = """
    SELECT code, vuln 
    FROM funcs 
    WHERE (end - start) BETWEEN ? AND ? 
    AND vuln = 1
    ORDER BY RANDOM()
    """
    
    # Get non-vulnerable functions
    non_vuln_query = """
    SELECT code, vuln 
    FROM funcs 
    WHERE (end - start) BETWEEN ? AND ? 
    AND vuln = 0
    ORDER BY RANDOM()
    """
    
    if max_per_class:
        vuln_query += f" LIMIT {max_per_class}"
        non_vuln_query += f" LIMIT {max_per_class}"
    
    # Execute queries
    cursor.execute(vuln_query, (min_lines, max_lines))
    vuln_functions = cursor.fetchall()
    
    cursor.execute(non_vuln_query, (min_lines, max_lines))
    non_vuln_functions = cursor.fetchall()
    
    conn.close()
    
    # Balance the dataset
    min_count = min(len(vuln_functions), len(non_vuln_functions))
    balanced_functions = (
        vuln_functions[:min_count] + 
        non_vuln_functions[:min_count]
    )
    
    print(f"Retrieved {min_count} vulnerable and {min_count} non-vulnerable functions")
    print(f"Total balanced dataset size: {len(balanced_functions)}")
    
    return balanced_functions

def prepare_function_tensors(functions, tokenizer_path='tokenizer.model', max_sequence_length=None):
    """
    Convert function code to tensors with proper padding.
    
    Args:
        functions: List of tuples (code, vulnerability_label)
        tokenizer_path: Path to SentencePiece tokenizer
        max_sequence_length: Maximum sequence length (auto-calculated if None)
    
    Returns:
        Tuple of (sequences_tensor, labels_tensor)
    """
    # Load tokenizer
    sp = spm.SentencePieceProcessor()
    sp.Load(tokenizer_path)
    
    # Extract code and labels
    codes = [func[0] for func in functions]
    labels = [func[1] for func in functions]
    
    # Tokenize code (split by lines and tokenize each line)
    sequences = []
    for code in codes:
        lines = code.split('\n')
        # Tokenize each line
        tokenized_lines = [sp.EncodeAsIds(line) for line in lines if line.strip()]
        sequences.append(tokenized_lines)
    
    # Calculate max dimensions if not provided
    if max_sequence_length is None:
        # Find the longest individual tokenized line
        max_sequence_length = max(
            max(len(line) for line in seq) if seq else 0 
            for seq in sequences
        )
        print(f"Auto-calculated max_sequence_length: {max_sequence_length}")
    
    # Find max number of lines in any function
    max_lines = max(len(seq) for seq in sequences)
    print(f"Max lines in any function: {max_lines}")
    
    # Pad sequences
    padded_sequences = []
    for seq in sequences:
        # Pad each line to max_sequence_length
        padded_lines = pad_sequences(
            seq, 
            maxlen=max_sequence_length, 
            padding='post', 
            truncating='post'
        )
        
        # Pad number of lines to max_lines
        if len(padded_lines) < max_lines:
            # Add empty lines (filled with zeros)
            empty_lines = np.zeros((max_lines - len(padded_lines), max_sequence_length), dtype=np.int32)
            padded_lines = np.vstack([padded_lines, empty_lines])
        elif len(padded_lines) > max_lines:
            # Truncate if too many lines
            padded_lines = padded_lines[:max_lines]
        
        padded_sequences.append(padded_lines)
    
    # Convert to tensors
    sequences_tensor = torch.tensor(padded_sequences, dtype=torch.long)
    labels_tensor = torch.tensor(labels, dtype=torch.float32)
    
    print(f"Sequences tensor shape: {sequences_tensor.shape}")
    print(f"Labels tensor shape: {labels_tensor.shape}")
    
    return sequences_tensor, labels_tensor

def create_balanced_dataset(db_path, min_lines=10, max_lines=50, 
                          max_per_class=None, test_size=0.2, 
                          tokenizer_path='tokenizer.model',
                          save_tensors=True, tensor_prefix='balanced_funcs'):
    """
    Complete pipeline to create balanced dataset from database.
    
    Args:
        db_path: Path to SQLite database
        min_lines: Minimum function length
        max_lines: Maximum function length
        max_per_class: Maximum functions per class
        test_size: Proportion for test set
        tokenizer_path: Path to tokenizer
        save_tensors: Whether to save tensors to files
        tensor_prefix: Prefix for saved tensor files
    
    Returns:
        Dictionary with train/test datasets and loaders
    """
    
    # Step 1: Query balanced functions
    functions = query_balanced_functions(
        db_path, min_lines, max_lines, max_per_class
    )
    
    # Step 2: Split into train/test
    train_functions, test_functions = train_test_split(
        functions, test_size=test_size, random_state=42, 
        stratify=[f[1] for f in functions]  # Stratify by vulnerability label
    )
    
    print(f"Train set size: {len(train_functions)}")
    print(f"Test set size: {len(test_functions)}")
    
    # Step 3: Convert to tensors
    train_sequences, train_labels = prepare_function_tensors(
        train_functions, tokenizer_path
    )
    test_sequences, test_labels = prepare_function_tensors(
        test_functions, tokenizer_path, 
        max_sequence_length=train_sequences.shape[-1]  # Use same max length
    )
    
    # Ensure same dimensions
    if test_sequences.shape[1] != train_sequences.shape[1]:
        # Pad test sequences to match train sequences
        max_lines = max(train_sequences.shape[1], test_sequences.shape[1])
        
        # Pad train sequences if needed
        if train_sequences.shape[1] < max_lines:
            padding = torch.zeros(
                train_sequences.shape[0], 
                max_lines - train_sequences.shape[1], 
                train_sequences.shape[2], 
                dtype=train_sequences.dtype
            )
            train_sequences = torch.cat([train_sequences, padding], dim=1)
        
        # Pad test sequences if needed
        if test_sequences.shape[1] < max_lines:
            padding = torch.zeros(
                test_sequences.shape[0], 
                max_lines - test_sequences.shape[1], 
                test_sequences.shape[2], 
                dtype=test_sequences.dtype
            )
            test_sequences = torch.cat([test_sequences, padding], dim=1)
    
    # Step 4: Create datasets and data loaders
    train_dataset = TensorDataset(train_sequences, train_labels)
    test_dataset = TensorDataset(test_sequences, test_labels)
    
    # Create data loaders
    batch_size = min(32, len(train_functions) // 4)  # Adaptive batch size
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, drop_last=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, drop_last=True
    )
    
    # Step 5: Save tensors if requested
    if save_tensors:
        torch.save(train_sequences, f'{tensor_prefix}_train_sequences.pt')
        torch.save(train_labels, f'{tensor_prefix}_train_labels.pt')
        torch.save(test_sequences, f'{tensor_prefix}_test_sequences.pt')
        torch.save(test_labels, f'{tensor_prefix}_test_labels.pt')
        print(f"Tensors saved with prefix: {tensor_prefix}")
    
    # Print statistics
    print(f"\nDataset Statistics:")
    print(f"Train - Vulnerable: {train_labels.sum().item():.0f}, Non-vulnerable: {(len(train_labels) - train_labels.sum()).item():.0f}")
    print(f"Test - Vulnerable: {test_labels.sum().item():.0f}, Non-vulnerable: {(len(test_labels) - test_labels.sum()).item():.0f}")
    print(f"Sequence shape: {train_sequences.shape}")
    print(f"Batch size: {batch_size}")
    
    return {
        'train_sequences': train_sequences,
        'train_labels': train_labels,
        'test_sequences': test_sequences,
        'test_labels': test_labels,
        'train_loader': train_loader,
        'test_loader': test_loader,
        'train_dataset': train_dataset,
        'test_dataset': test_dataset
    }

# Example usage
if __name__ == "__main__":
    # Use your database files
    datasets = {}
    
    # Process Java database
    if os.path.exists('java_10+.db'):
        print("Processing Java database...")
        datasets['java'] = create_balanced_dataset(
            'java_10+.db',
            min_lines=10,
            max_lines=50,
            max_per_class=5000,  # Limit to 5000 per class
            tensor_prefix='java_balanced_funcs'
        )
    
    # Process C database
    if os.path.exists('c_10+.db'):
        print("\nProcessing C database...")
        datasets['c'] = create_balanced_dataset(
            'c_10+.db',
            min_lines=10,
            max_lines=50,
            max_per_class=5000,
            tensor_prefix='c_balanced_funcs'
        )
    
    print("\nDataset creation complete!")

ModuleNotFoundError: No module named 'keras'